In [1]:
from pathlib import Path
import pandas as pd
import re

from lzn.data_processing.parse_dftsp_grepscf_to_df import parse_grepscf
from lzn.universal import *

from time import asctime

from pandas.core.tools.times import to_time

## def function: Combine energy calculation
combine energy per TiO2 unit $$E_{comb} = E_{n}/n - E_1$$

In [2]:
def get_combine_energy(df: pd.DataFrame) -> pd.DataFrame:
    # format the output df
    column_comb = [x.split(" ")[0]+" eV" for x in df.columns]
    index_comb = df.index
    df_comb = pd.DataFrame(columns=column_comb, index=index_comb)

    # calculate the values
    for i in [x.split(" ")[0] for x in df.columns]:
        n = re.search(r'^(\d+)[a-z]+', i).group(1)
        n = int(n)
        # print(n)
        E_i_name = i + " au"
        dE_i_name = i + " eV"
        df_comb.loc[:,dE_i_name] = \
            (df.loc[:, E_i_name] / n - df.loc[:, "1a au"]) * hartree_to_eV

    return df_comb



# Optimization methods own energies

### MLIP energies (LOAD from chen_1-64_5mlip_a.csv generated by 260122)

In [3]:
# mlE_dir = to_path("~/TiO2/tio2_ChenDixon_out")
# modals = [x for x in mlE_dir.iterdir() if x.is_dir()]
# mlE_path = [x / "energies.csv" for x in modals]
#
# # print(modals)
# # print(mlE_path)
#
# df_ml = [pd.read_csv(x) for x in mlE_path]
#
#
#
# # sort file values like: "40a.xyz", "5b.xyz", "64b.xyz" by number then name
# def sort_rows(df: pd.DataFrame, col_name: str = "file"):
#     if col_name == "index":
#         tmp = df.index.str.extract(r"^(\d+)([a-z])", expand=True)
#         tmp.loc[:,"ind"] = df.index
#         tmp.set_index("ind", inplace=True)
#         tmp.loc[:,0] = tmp.loc[:,0].astype(int)
#         df = pd.concat([df, tmp], axis=1)
#         df_sorted = (
#             df.sort_values([0, 1])
#               .drop(columns=[0, 1])
#         )
#     else:
#         tmp = df[col_name].str.extract(r"^(\d+)([a-z])", expand=True)  # number, letter
#         df_sorted = (
#             df.assign(_n=tmp[0].astype(int), _s=tmp[1].str.lower())
#               .sort_values(["_n", "_s"])
#               .drop(columns=["_n", "_s"])
#               .reset_index(drop=True)
#         )
#     return df_sorted
#
# # substract .xyz in file column
# def rename_sub_xyz(df: pd.DataFrame):
#     df.loc[:,"file"] = df.loc[:, "file"].str.split(".").str[0]
#     return df
#
#
# df_ml_sorted = []
# for i, df in enumerate(df_ml):
#     # sort and reformat file column
#     df_sorted = sort_rows(df)
#     df_sorted = rename_sub_xyz(df_sorted)
#
#     # only keep E_final and rename it to the modal name
#     df_sorted = df_sorted.drop(columns=["E_initial", "dE"])
#     modal_name = modals[i].stem
#     df_sorted = df_sorted.rename(columns={"E_final": modal_name})
#
#     # rename file column to Method, and set it as index
#     df_sorted.rename(columns={"file": "Method"}, inplace=True)
#     df_sorted.set_index("Method", inplace=True)
#
#     df_ml_sorted.append(df_sorted)
#
#
# print(len(df_ml_sorted))
# df_ml_tot = pd.concat(df_ml_sorted, axis=1)
# df_ml_tot = sort_rows(df_ml_tot, col_name="index")
# df_ml_tot = df_ml_tot.T
#
# # print(df_ml_tot.columns.values)
# df_ml_au = df_ml_tot.copy()
# df_ml_au.columns = [x+" au" for x in df_ml_tot.columns]
# df_ml_au = df_ml_au / hartree_to_eV
#
# df_ml_au

In [4]:
path_mlip_df = to_path("~/organized/901_data/chen_1-64_5mlip_r.csv")
df_mlip = pd.read_csv(path_mlip_df)
df_mlip.set_index("id", inplace=True)

In [5]:
df_mlip

,ref_eV,matpes_pbe,omat24,omol25_low,mpa,matpes_r2scan
id,,,,,,
1a,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5a,-4.026534,-3.909440,-3.918639,-4.299219,-3.973055,-4.321299
5b,-4.024956,-3.903593,-3.921683,-4.336719,-3.967011,-4.334269
8a,-4.621538,-4.449787,-4.461456,-4.964844,-4.502728,-4.951170
8b,-4.607548,-4.432922,-4.448328,-4.972656,-4.492523,-4.937876
12a,-4.924655,-4.739760,-4.750633,-5.324219,-4.787299,-5.287616
12b,-4.831207,-4.693429,-4.713351,-5.342448,-4.775003,-5.277019
12c,-4.796835,-4.659173,-4.669095,-5.287760,-4.730518,-5.230607
12d,-4.616196,-4.425592,-4.432409,-4.902344,-4.465445,-4.893522


### DFT opt energies

In [6]:
path_ = to_path("~/TiO2/dft_energies_tot_opt.csv")
df_dft = pd.read_csv(path_)

# combine Method and Basis, and delete 3rd, 4th columns
df_dft.loc[:,"id"] = df_dft.loc[:,"Method"] + '_' + df_dft.loc[:,"Basis"]
df_dft.drop(columns=["Basis", "Method"], inplace=True)

# copy pbe0svp optimized pbe0tzvp method to pbe0tzvp 1/ Except that 12a calculated exactly
# df_dft.iloc[2, 8:12] = df_dft.iloc[3, 8:12]
df_dft.drop([3], inplace=True)

df_dft.drop(columns=["Dispersion","original0/optimized1"], inplace=True)
df_dft.set_index("id", inplace=True)

df_dft

,1a au,5a au,5b au,8a au,8b au,12a au,12b au,12c au,12d au,40a au,40b au,40c au,64a au,64b au
id,,,,,,,,,,,,,,
article_article,-999.890386,-5000.191792,-5000.191502,-8000.481795,-8000.477682,-12000.856365,-12000.815155,-12000.799997,-12000.720337,-40003.545213,-40003.536524,-40003.423463,-64005.820291,-64005.817521
pbe0_def2svp,-999.357624,-4997.617272,-4997.628051,-7996.396703,-7996.401048,-11994.753940,-11994.730495,-11994.702984,-11994.578923,NaN,NaN,NaN,NaN,NaN
pbe0_def2tzvp,-999.652860,-4999.060280,-4999.064222,-7998.690782,-7998.690928,-11998.190199,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pbe_def2svp,-999.380991,-4997.691722,-4997.699375,-7996.501134,-7996.504978,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pbe_def2tzvp,-999.678333,-4999.131012,-4999.133300,-7998.788779,-7998.788667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_dft = get_combine_energy(df_dft)
df_dft = df_dft.transpose()
df_dft.index = [i_.split(" ")[0] for i_ in df_dft.index]
df_dft.drop(columns=["article_article"], inplace=True)
df_dft

id,pbe0_def2svp,pbe0_def2tzvp,pbe_def2svp,pbe_def2tzvp
1a,0.0,0.0,0.0,0.0
5a,-4.512489,-4.331947,-4.281816,-4.023739
5b,-4.571154,-4.3534,-4.323466,-4.036192
8a,-5.22362,-4.99296,-4.942982,-4.633137
8b,-5.238399,-4.993457,-4.95606,-4.632757
12a,-5.58391,-5.342232,NaN,NaN
12b,-5.530746,NaN,NaN,NaN
12c,-5.468361,NaN,NaN,NaN
12d,-5.187037,NaN,NaN,NaN
40a,NaN,NaN,NaN,NaN


### combine two df and calculate df_comb

In [8]:
# concat two df
df_ownE = pd.concat([df_mlip, df_dft], axis=1)

df_ownE

,ref_eV,matpes_pbe,omat24,omol25_low,mpa,matpes_r2scan,pbe0_def2svp,pbe0_def2tzvp,pbe_def2svp,pbe_def2tzvp
1a,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
5a,-4.026534,-3.909440,-3.918639,-4.299219,-3.973055,-4.321299,-4.512489,-4.331947,-4.281816,-4.023739
5b,-4.024956,-3.903593,-3.921683,-4.336719,-3.967011,-4.334269,-4.571154,-4.3534,-4.323466,-4.036192
8a,-4.621538,-4.449787,-4.461456,-4.964844,-4.502728,-4.951170,-5.22362,-4.99296,-4.942982,-4.633137
8b,-4.607548,-4.432922,-4.448328,-4.972656,-4.492523,-4.937876,-5.238399,-4.993457,-4.95606,-4.632757
12a,-4.924655,-4.739760,-4.750633,-5.324219,-4.787299,-5.287616,-5.58391,-5.342232,NaN,NaN
12b,-4.831207,-4.693429,-4.713351,-5.342448,-4.775003,-5.277019,-5.530746,NaN,NaN,NaN
12c,-4.796835,-4.659173,-4.669095,-5.287760,-4.730518,-5.230607,-5.468361,NaN,NaN,NaN
12d,-4.616196,-4.425592,-4.432409,-4.902344,-4.465445,-4.893522,-5.187037,NaN,NaN,NaN
40a,-5.394503,-5.159000,-5.165060,-5.849219,-5.213998,-5.803492,NaN,NaN,NaN,NaN


# tzvp SP energies

### read all sp energies from grepscf to df

In [9]:
# read grepscf txt files
folder = to_path("~/TiO2/all_sp")
path_1 = folder / "all1_sp" / "energies.txt"
path_5 = folder / "all5_sp" / "energies.txt"
path_8 = folder / "all8_sp" / "energies.txt"

df_1 = parse_grepscf(path_1)
df_5 = parse_grepscf(path_5)
df_8 = parse_grepscf(path_8)

df_list = [df_1, df_5, df_8]

In [10]:
df_8

,structure,opt method,sp method,energy_au
0,8a,article_article,pbe0_def2tzvp,-7998.680535
1,8a,matpes_pbe,pbe0_def2tzvp,-7998.675239
2,8a,matpes_r2scan,pbe0_def2tzvp,-7998.684010
3,8a,mpa_mpa,pbe0_def2tzvp,-7998.675920
4,8a,omat24_omat24,pbe0_def2tzvp,-7998.675645
5,8a,omol25_low,pbe0_def2tzvp,-7998.689734
6,8a,pbe0_def2svp,pbe0_def2tzvp,-7998.689023
7,8a,pbe0_def2tzvp,pbe0_def2tzvp,-7998.690782
8,8a,pbe_def2svp,pbe0_def2tzvp,-7998.688561
9,8a,article_article,pbe_def2tzvp,-7998.787842


### merge to two groups: df_pbe0, df_pbe

In [11]:
# split to two groups: pbe0, pbe
df_list_pbe0 = [df[df["sp method"] == "pbe0_def2tzvp"] for df in df_list]
df_list_pbe = [df[df["sp method"] == "pbe_def2tzvp"] for df in df_list]


# matching and sorting to methods list
def match_and_sort(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract rows where 'opt method' matches methods list,
    sorted by methods order.
    """
    methods = ['article_article', 'pbe0_def2svp', 'pbe0_def2tzvp', 'pbe_def2svp', 'pbe_def2tzvp', 'matpes_pbe', 'omat24_omat24', 'omol25_low', 'mpa_mpa', 'matpes_r2scan']

    # Filter rows matching methods
    mask = df['opt method'].isin(methods)
    filtered = df[mask].copy()

    # Sort by methods order (stable sort)
    filtered['sort_key'] = filtered['opt method'].map({m: i for i, m in enumerate(methods)})
    result = filtered.sort_values('sort_key').drop(columns='sort_key')

    return result


df_list_pbe0 = [match_and_sort(df) for df in df_list_pbe0]
df_list_pbe = [match_and_sort(df) for df in df_list_pbe]

df_list_pbe0 = [df.drop(columns=["sp method"]) for df in df_list_pbe0]
df_list_pbe = [df.drop(columns=["sp method"]) for df in df_list_pbe]

for i in df_list_pbe0+df_list_pbe:
    print(len(i.index))

# print(df_list_pbe0[2])
df_list_pbe0[0]

10
20
20
10
20
20


,structure,opt method,energy_au
0,1a,article_article,-999.651649
6,1a,pbe0_def2svp,-999.652513
7,1a,pbe0_def2tzvp,-999.652860
8,1a,pbe_def2svp,-999.652330
9,1a,pbe_def2tzvp,-999.651774
1,1a,matpes_pbe,-999.650344
4,1a,omat24_omat24,-999.650574
5,1a,omol25_low,-999.652064
3,1a,mpa_mpa,-999.650206
2,1a,matpes_r2scan,-999.651591


In [12]:
df_list_pbe0[2].reset_index(drop=True)

,structure,opt method,energy_au
0,8a,article_article,-7998.680535
1,8b,article_article,-7998.680393
2,8b,pbe0_def2svp,-7998.689135
3,8a,pbe0_def2svp,-7998.689023
4,8a,pbe0_def2tzvp,-7998.690782
5,8b,pbe0_def2tzvp,-7998.690928
6,8b,pbe_def2svp,-7998.688511
7,8a,pbe_def2svp,-7998.688561
8,8b,pbe_def2tzvp,-7998.616452
9,8a,pbe_def2tzvp,-7998.618073


In [13]:
def reformat_dftsp_df(df: pd.DataFrame) -> pd.DataFrame:
    """separate a and b"""
    df_out = pd.DataFrame(index=['article_article', 'pbe0_def2svp', 'pbe0_def2tzvp', 'pbe_def2svp', 'pbe_def2tzvp', 'matpes_pbe', 'omat24_omat24', 'omol25_low', 'mpa_mpa', 'matpes_r2scan'])
    df = df.reset_index(drop=True)

    for i in range(len(df.index)):
        structure = df.loc[i, "structure"]
        energy = df.loc[i, "energy_au"]

        new_column_name = structure + " au"
        new_index = df.loc[i, "opt method"]

        df_out.loc[new_index, new_column_name] = energy

    return df_out

reformat_dftsp_df(df_list_pbe0[2])

,8a au,8b au
article_article,-7998.680535,-7998.680393
pbe0_def2svp,-7998.689023,-7998.689135
pbe0_def2tzvp,-7998.690782,-7998.690928
pbe_def2svp,-7998.688561,-7998.688511
pbe_def2tzvp,-7998.618073,-7998.616452
matpes_pbe,-7998.675239,-7998.674781
omat24_omat24,-7998.675645,-7998.675534
omol25_low,-7998.689734,-7998.689868
mpa_mpa,-7998.675920,-7998.675791
matpes_r2scan,-7998.684010,-7998.683693


In [14]:
df_list_pbe0_reformat = [reformat_dftsp_df(df) for df in df_list_pbe0]
df_list_pbe_reformat = [reformat_dftsp_df(df) for df in df_list_pbe]

df_list_pbe0_reformat[1]

,5a au,5b au
article_article,-4999.054146,-4999.057686
pbe0_def2svp,-4999.058998,-4999.063063
pbe0_def2tzvp,-4999.060280,-4999.064222
pbe_def2svp,-4999.058532,-4999.062800
pbe_def2tzvp,-4999.055563,-4999.059600
matpes_pbe,-4999.050827,-4999.054039
omat24_omat24,-4999.051386,-4999.054621
omol25_low,-4999.059019,-4999.063340
mpa_mpa,-4999.051394,-4999.054309
matpes_r2scan,-4999.055888,-4999.059767


In [15]:
df_pbe_tot = pd.concat(df_list_pbe_reformat, axis=1)
df_pbe0_tot = pd.concat(df_list_pbe0_reformat, axis=1)

df_pbe0_tot

,1a au,5a au,5b au,8a au,8b au
article_article,-999.651649,-4999.054146,-4999.057686,-7998.680535,-7998.680393
pbe0_def2svp,-999.652513,-4999.058998,-4999.063063,-7998.689023,-7998.689135
pbe0_def2tzvp,-999.652860,-4999.060280,-4999.064222,-7998.690782,-7998.690928
pbe_def2svp,-999.652330,-4999.058532,-4999.062800,-7998.688561,-7998.688511
pbe_def2tzvp,-999.651774,-4999.055563,-4999.059600,-7998.618073,-7998.616452
matpes_pbe,-999.650344,-4999.050827,-4999.054039,-7998.675239,-7998.674781
omat24_omat24,-999.650574,-4999.051386,-4999.054621,-7998.675645,-7998.675534
omol25_low,-999.652064,-4999.059019,-4999.063340,-7998.689734,-7998.689868
mpa_mpa,-999.650206,-4999.051394,-4999.054309,-7998.675920,-7998.675791
matpes_r2scan,-999.651591,-4999.055888,-4999.059767,-7998.684010,-7998.683693


In [16]:
df_pbe_tot

,1a au,5a au,5b au,8a au,8b au
article_article,-999.678306,-4999.130072,-4999.132496,-7998.787842,-7998.787465
pbe0_def2svp,-999.676157,-4999.121563,-4999.123890,-7998.774426,-7998.773949
pbe0_def2tzvp,-999.677308,-4999.126568,-4999.128925,-7998.716403,-7998.714372
pbe_def2svp,-999.677931,-4999.129887,-4999.132172,-7998.787115,-7998.786961
pbe_def2tzvp,-999.678333,-4999.131012,-4999.133300,-7998.788779,-7998.788667
matpes_pbe,-999.677976,-4999.129773,-4999.131934,-7998.786960,-7998.786760
omat24_omat24,-999.678091,-4999.130033,-4999.132106,-7998.787062,-7998.786959
omol25_low,-999.677023,-4999.127026,-4999.130478,-7998.784245,-7998.784818
mpa_mpa,-999.677985,-4999.130069,-4999.131994,-7998.787075,-7998.787039
matpes_r2scan,-999.678178,-4999.129994,-4999.132919,-7998.788403,-7998.788164


### get comb energy df

In [17]:
df_pbe0_comb = get_combine_energy(df_pbe0_tot)
df_pbe_comb = get_combine_energy(df_pbe_tot)

df_pbe0_comb

,1a eV,5a eV,5b eV,8a eV,8b eV
article_article,0.0,-4.33152,-4.350785,-4.99106,-4.990576
pbe0_def2svp,0.0,-4.334425,-4.356549,-4.996429,-4.996813
pbe0_def2tzvp,0.0,-4.331947,-4.3534,-4.99296,-4.993457
pbe_def2svp,0.0,-4.336844,-4.36007,-4.999812,-4.999642
pbe_def2tzvp,0.0,-4.335816,-4.357788,-4.775186,-4.769672
matpes_pbe,0.0,-4.348959,-4.366442,-5.008549,-5.006991
omat24_omat24,0.0,-4.345745,-4.363349,-5.003675,-5.003297
omol25_low,0.0,-4.346739,-4.370252,-5.011048,-5.011503
mpa_mpa,0.0,-4.355792,-4.371655,-5.014613,-5.014175
matpes_r2scan,0.0,-4.342568,-4.36368,-5.004447,-5.00337


# Results

Energies from theirself methods

In [18]:
general_save = to_path("~/organized/901_data")

In [19]:
df_ownE.to_csv(general_save / "chen1-64_allmethods_r.csv")
df_ownE

,ref_eV,matpes_pbe,omat24,omol25_low,mpa,matpes_r2scan,pbe0_def2svp,pbe0_def2tzvp,pbe_def2svp,pbe_def2tzvp
1a,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
5a,-4.026534,-3.909440,-3.918639,-4.299219,-3.973055,-4.321299,-4.512489,-4.331947,-4.281816,-4.023739
5b,-4.024956,-3.903593,-3.921683,-4.336719,-3.967011,-4.334269,-4.571154,-4.3534,-4.323466,-4.036192
8a,-4.621538,-4.449787,-4.461456,-4.964844,-4.502728,-4.951170,-5.22362,-4.99296,-4.942982,-4.633137
8b,-4.607548,-4.432922,-4.448328,-4.972656,-4.492523,-4.937876,-5.238399,-4.993457,-4.95606,-4.632757
12a,-4.924655,-4.739760,-4.750633,-5.324219,-4.787299,-5.287616,-5.58391,-5.342232,NaN,NaN
12b,-4.831207,-4.693429,-4.713351,-5.342448,-4.775003,-5.277019,-5.530746,NaN,NaN,NaN
12c,-4.796835,-4.659173,-4.669095,-5.287760,-4.730518,-5.230607,-5.468361,NaN,NaN,NaN
12d,-4.616196,-4.425592,-4.432409,-4.902344,-4.465445,-4.893522,-5.187037,NaN,NaN,NaN
40a,-5.394503,-5.159000,-5.165060,-5.849219,-5.213998,-5.803492,NaN,NaN,NaN,NaN


pbe0 tzvp SP energies

In [20]:
df_pbe0_tot

,1a au,5a au,5b au,8a au,8b au
article_article,-999.651649,-4999.054146,-4999.057686,-7998.680535,-7998.680393
pbe0_def2svp,-999.652513,-4999.058998,-4999.063063,-7998.689023,-7998.689135
pbe0_def2tzvp,-999.652860,-4999.060280,-4999.064222,-7998.690782,-7998.690928
pbe_def2svp,-999.652330,-4999.058532,-4999.062800,-7998.688561,-7998.688511
pbe_def2tzvp,-999.651774,-4999.055563,-4999.059600,-7998.618073,-7998.616452
matpes_pbe,-999.650344,-4999.050827,-4999.054039,-7998.675239,-7998.674781
omat24_omat24,-999.650574,-4999.051386,-4999.054621,-7998.675645,-7998.675534
omol25_low,-999.652064,-4999.059019,-4999.063340,-7998.689734,-7998.689868
mpa_mpa,-999.650206,-4999.051394,-4999.054309,-7998.675920,-7998.675791
matpes_r2scan,-999.651591,-4999.055888,-4999.059767,-7998.684010,-7998.683693


In [21]:
df_pbe0_comb = df_pbe0_comb.transpose()
df_pbe0_comb.to_csv(general_save / "chen1-8_allmethods_pbe0spr.csv")
df_pbe0_comb

,article_article,pbe0_def2svp,pbe0_def2tzvp,pbe_def2svp,pbe_def2tzvp,matpes_pbe,omat24_omat24,omol25_low,mpa_mpa,matpes_r2scan
1a eV,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5a eV,-4.33152,-4.334425,-4.331947,-4.336844,-4.335816,-4.348959,-4.345745,-4.346739,-4.355792,-4.342568
5b eV,-4.350785,-4.356549,-4.3534,-4.36007,-4.357788,-4.366442,-4.363349,-4.370252,-4.371655,-4.36368
8a eV,-4.99106,-4.996429,-4.99296,-4.999812,-4.775186,-5.008549,-5.003675,-5.011048,-5.014613,-5.004447
8b eV,-4.990576,-4.996813,-4.993457,-4.999642,-4.769672,-5.006991,-5.003297,-5.011503,-5.014175,-5.00337


pbe tzvp SP energies

In [22]:

df_pbe_tot

,1a au,5a au,5b au,8a au,8b au
article_article,-999.678306,-4999.130072,-4999.132496,-7998.787842,-7998.787465
pbe0_def2svp,-999.676157,-4999.121563,-4999.123890,-7998.774426,-7998.773949
pbe0_def2tzvp,-999.677308,-4999.126568,-4999.128925,-7998.716403,-7998.714372
pbe_def2svp,-999.677931,-4999.129887,-4999.132172,-7998.787115,-7998.786961
pbe_def2tzvp,-999.678333,-4999.131012,-4999.133300,-7998.788779,-7998.788667
matpes_pbe,-999.677976,-4999.129773,-4999.131934,-7998.786960,-7998.786760
omat24_omat24,-999.678091,-4999.130033,-4999.132106,-7998.787062,-7998.786959
omol25_low,-999.677023,-4999.127026,-4999.130478,-7998.784245,-7998.784818
mpa_mpa,-999.677985,-4999.130069,-4999.131994,-7998.787075,-7998.787039
matpes_r2scan,-999.678178,-4999.129994,-4999.132919,-7998.788403,-7998.788164


In [23]:
df_pbe_comb = df_pbe_comb.transpose()
df_pbe_comb.to_csv(general_save / "chen1-8_allmethods_pbespr.csv")
df_pbe_comb

,article_article,pbe0_def2svp,pbe0_def2tzvp,pbe_def2svp,pbe_def2tzvp,matpes_pbe,omat24_omat24,omol25_low,mpa_mpa,matpes_r2scan
1a eV,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5a eV,-4.019342,-4.03153,-4.027441,-4.028564,-4.023739,-4.026717,-4.025001,-4.037689,-4.028068,-4.022425
5b eV,-4.032536,-4.044194,-4.040271,-4.040997,-4.036192,-4.038474,-4.036285,-4.056474,-4.038545,-4.03834
8a eV,-4.630668,-4.643531,-4.414845,-4.638421,-4.633137,-4.636667,-4.633889,-4.65336,-4.636805,-4.636084
8b eV,-4.629385,-4.641908,-4.407937,-4.6379,-4.632757,-4.635988,-4.633539,-4.655307,-4.636683,-4.63527
